# Merchant fraud: tier classifier and ranking penalty

This notebook has two parts.

1. **Tier classifier.** It uses the same 30 / 50 / 70 cutoffs as `fraud_classifier.ipynb`, applied to the 114 scored merchant-days. The predictors describe how unusual the day was for that merchant: revenue, count, average and largest transaction relative to the merchant's usual. This part tests how far the small merchant sample can go.
2. **Penalty.** Expected fraud loss in dollars is subtracted from each merchant's estimated BNPL revenue, and the `merchant_ranking_v2` score is recomputed. This closes the "Fraud — expected loss should come off the score in dollars" item in that notebook's limitations.

**Run order:** `scripts/join_external_sa2.py` → `01_data_quality_eda.ipynb` → `merchant_ranking_v2.ipynb` → `merchant_risk.ipynb` → this notebook.

## 1. Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession, Window, functions as F
from scipy.stats import spearmanr
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, roc_auc_score
from xgboost import XGBClassifier

# Works when launched from the repository, project2, or notebooks directory.
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if p.name == "project2"), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd() / "project2"
PROCESSED_PATH = PROJECT_ROOT / "data" / "tables" / "processed"
RANKING_IN = PROCESSED_PATH / "merchant_ranking_v2.parquet"
RISK_IN = PROCESSED_PATH / "merchant_risk"
for path in (RANKING_IN, RISK_IN):
    if not path.exists():
        raise FileNotFoundError(f"{path} is missing - see the run order above.")
RANDOM_STATE = 42
CUTOFFS = [30, 50, 70]
# Fraud scores exist only in this window.
START, END = "2021-02-28", "2022-02-27"
# Without a cap, XGBoost oversubscribes the CPU alongside Spark and slows to a crawl.
XGB_THREADS = 4
spark = SparkSession.builder.config("spark.driver.memory", "6g").getOrCreate()

## 2. Merchant-day features

Every merchant-day in the scoring window is compared with that merchant's usual (median) behaviour over the same window.

In [ ]:
trans = (
    spark.read.parquet(str(PROCESSED_PATH / "transactions_clean"))
    .withColumn("order_date", F.to_date("order_datetime"))
    .where(F.col("order_date").between(START, END))
)
merchant_w = Window.partitionBy("merchant_abn")
merchant_txn = trans.groupBy("merchant_abn").agg(
    F.percentile_approx("dollar_value", 0.5).alias("merchant_txn_median"),
    F.mean("dollar_value").alias("merchant_txn_mean"),
    F.stddev("dollar_value").alias("merchant_txn_std"))
merchant_day = (
    trans.groupBy("merchant_abn", "order_date")
    .agg(F.sum("dollar_value").alias("revenue"), F.count("*").alias("txn_count"),
         F.mean("dollar_value").alias("avg_value"), F.max("dollar_value").alias("max_value"))
    .withColumn("typical_revenue", F.percentile_approx("revenue", 0.5).over(merchant_w))
    .withColumn("typical_count", F.percentile_approx("txn_count", 0.5).over(merchant_w))
    .withColumn("revenue_mean", F.mean("revenue").over(merchant_w))
    .withColumn("revenue_std", F.stddev("revenue").over(merchant_w))
    .withColumn("active_days", F.count("*").over(merchant_w))
    .join(merchant_txn, "merchant_abn")
)

merchant_scores = (spark.read.parquet(str(PROCESSED_PATH / "merchant_fraud_probability"))
                   .where(F.col("fraud_probability").isNotNull())
                   .withColumn("order_date", F.to_date("order_datetime")).drop("order_datetime"))
days = merchant_scores.join(merchant_day, ["merchant_abn", "order_date"], "inner").toPandas()
assert len(days) == merchant_scores.count(), "every scored merchant-day should match transactions"

days["revenue_ratio"] = days["revenue"] / days["typical_revenue"]
days["count_ratio"] = days["txn_count"] / days["typical_count"]
days["avg_value_ratio"] = days["avg_value"] / days["merchant_txn_median"]
days["max_value_ratio"] = days["max_value"] / days["merchant_txn_median"]
# Share of the day's revenue that came from its single largest transaction.
days["max_share_of_day"] = days["max_value"] / days["revenue"]
# A merchant with a single active day has no spread; treat it as "not unusual".
days["revenue_z"] = ((days["revenue"] - days["revenue_mean"]) / days["revenue_std"]).fillna(0)
days["max_value_z"] = ((days["max_value"] - days["merchant_txn_mean"])
                       / days["merchant_txn_std"]).fillna(0)
days["merchant_txn_cv"] = (days["merchant_txn_std"] / days["merchant_txn_mean"]).fillna(0)
days["log_revenue"] = np.log(days["revenue"])
days["log_typical_revenue"] = np.log(days["typical_revenue"])
days["log_active_days"] = np.log(days["active_days"])
p = days["fraud_probability"]
days["tier"] = np.select([p >= 70, p >= 50, p >= 30], [3, 2, 1], 0)

FEATURES = ["revenue_ratio", "count_ratio", "avg_value_ratio", "max_value_ratio",
            "max_share_of_day", "revenue_z", "max_value_z", "merchant_txn_cv",
            "log_revenue", "log_typical_revenue", "log_active_days"]
TIER_NAMES = {0: "<30", 1: "30-50", 2: "50-70", 3: ">=70"}
print(f"{len(days)} scored merchant-days across {days['merchant_abn'].nunique()} merchants")
days.groupby("tier")[FEATURES].median().rename(index=TIER_NAMES).T.round(2) \
    .assign(**{"Spearman with score": [spearmanr(days[c], p).statistic for c in FEATURES]}).round(2)

- **Higher tiers are single oversized transactions.** Moving up the tiers, the day's count stays near normal, while the average and largest transaction grow relative to the merchant's usual.
- **At ≥ 70, one transaction is the whole day.** The median `max_share_of_day` is 1.0.
- **This matches the consumer side.** There, the score tracked transaction size relative to the user's usual.

## 3. Cumulative classifiers, leave-one-merchant-out

With only 114 rows, one test split would be too noisy. Instead each merchant is held out in turn: it is scored by a model trained on the other 60 merchants.

Two model types are compared:
- **Logistic regression** (L2-regularised, standardised features)
- **Shallow XGBoost** (depth 2)

The PR-AUC of a random ranking equals the prevalence.

In [ ]:
MODELS = {
    "logistic": lambda: make_pipeline(StandardScaler(), LogisticRegression(C=0.5, max_iter=5000)),
    "xgboost (depth 2)": lambda: XGBClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=2, subsample=0.8,
        n_jobs=XGB_THREADS, random_state=RANDOM_STATE),
}
groups = days["merchant_abn"]
oof, rows = {}, []
for k in CUTOFFS:
    y = (days["fraud_probability"] >= k).astype(int)
    for name, make in MODELS.items():
        pred = cross_val_predict(make(), days[FEATURES], y, groups=groups,
                                 cv=LeaveOneGroupOut(), method="predict_proba")[:, 1]
        oof[(k, name)] = pred
        rows.append({"cutoff": f">={k}", "model": name, "positive days": int(y.sum()),
                     "positive merchants": days.loc[y == 1, "merchant_abn"].nunique(),
                     "prevalence": y.mean(), "PR-AUC": average_precision_score(y, pred),
                     "ROC-AUC": roc_auc_score(y, pred)})
pd.DataFrame(rows).round(3)

In [ ]:
coefs = pd.DataFrame({
    f">={k}": MODELS["logistic"]().fit(days[FEATURES], (days["fraud_probability"] >= k).astype(int))[-1].coef_[0]
    for k in CUTOFFS}, index=FEATURES)
coefs.round(2).sort_values(">=50")

- **≥ 30 and ≥ 50 are learnable.** Both are well above the random baseline on merchants the model never saw.
- **≥ 70 is not.** There are 7 days, each from a different merchant, so every fold predicts a merchant from 6 examples. Its numbers are unstable and shouldn't be reported as a model result.
- **Scope.** The model is trained only on scored days. Like the consumer classifier, it ranks days that were already selected for scoring. It does not give a fraud probability for an ordinary day.

For the penalty below, the **observed** scores are used directly rather than these predictions. Unscored merchants are covered by the `suspect` flag from `merchant_risk.ipynb`.

## 4. Expected fraud loss per merchant

When a BNPL transaction is fraudulent, the firm has paid the merchant and is not repaid. The loss is up to the transaction's **full value**, not the take-rate fee. For each transaction in the scoring window:

- **Merchant-day score:** `p_merchant`, the score for that merchant on that day.
- **Consumer user-day score:** `p_consumer`. On shared days it is split in proportion to dollar value, so the day's largest transaction carries most of it. This matches `fraud_classifier.ipynb`, where a day's score follows its largest transaction.
- **Combined:** P(fraud) = 1 − (1 − p_merchant)(1 − p_consumer).
- **Expected loss** = `LOSS_SEVERITY` × P(fraud) × `dollar_value`.

A score only counts if it is at least `MIN_SCORE`. Unscored transactions contribute 0, so the expected loss is a **lower bound**, because unscored does not mean clean.

Fraud scores exist only in the window, while `estimated_bnpl_revenue` in the ranking covers every transaction. So the loss enters as a **rate**:

`loss_rate` = expected loss ÷ value, both within the window.

Then:

`fraud_adjusted_revenue` = `estimated_bnpl_revenue` − `loss_rate` × `total_value`.

Losses are computed below for several `MIN_SCORE` values at once, so §5 can compare them.

In [ ]:
MIN_SCORE_OPTIONS = [0, 30, 50]

consumer_scores = (spark.read.parquet(str(PROCESSED_PATH / "consumer_fraud_probability"))
                   .where(F.col("fraud_probability").isNotNull())
                   .withColumn("order_date", F.to_date("order_datetime"))
                   .select("user_id", "order_date", F.col("fraud_probability").alias("consumer_score")))
merchant_p = merchant_scores.select("merchant_abn", "order_date",
                                    F.col("fraud_probability").alias("merchant_score"))

user_day_w = Window.partitionBy("user_id", "order_date")
txn = (
    trans
    .join(consumer_scores, ["user_id", "order_date"], "left")
    .join(merchant_p, ["merchant_abn", "order_date"], "left")
    .withColumn("value_share_of_user_day",
                F.col("dollar_value") / F.sum("dollar_value").over(user_day_w))
)

def p_if_at_least(score_col, min_score):
    return F.when(F.col(score_col) >= min_score, F.col(score_col) / 100).otherwise(0.0)

loss_aggs = [F.sum("dollar_value").alias("window_value")]
for m in MIN_SCORE_OPTIONS:
    p_consumer = p_if_at_least("consumer_score", m) * F.col("value_share_of_user_day")
    p_merchant = p_if_at_least("merchant_score", m)
    p_fraud = 1 - (1 - p_consumer) * (1 - p_merchant)
    loss_aggs += [F.sum(p_fraud * F.col("dollar_value")).alias(f"loss_min{m}"),
                  F.sum(p_consumer * F.col("dollar_value")).alias(f"consumer_loss_min{m}")]
merchant_loss = txn.groupBy("merchant_abn").agg(*loss_aggs).toPandas()

for m in MIN_SCORE_OPTIONS:
    total = merchant_loss[f"loss_min{m}"].sum()
    print(f"MIN_SCORE {m:>2}: expected loss in window ${total / 1e6:6.2f}M "
          f"({merchant_loss[f'consumer_loss_min{m}'].sum() / total:.0%} from consumer scores), "
          f"{(merchant_loss[f'loss_min{m}'] > 0).sum():,} merchants affected")

Consumer scores dominate at every setting. Since a consumer score follows transaction size, merchants that sell expensive items collect the most scored transactions. At `MIN_SCORE` 0, even the 8–15% "scored low" days count. A 10% chance of losing a $10K principal is about $1K of expected loss, against a few hundred dollars of fee revenue on the same sale.

## 5. Apply the penalty to the v2 ranking

v2 turns each feature into a percentile rank. Revenue is shrunk towards 50 for thin evidence, using `n / (n + 300)`, and carries 70% of the weight.

The adjusted score is v2's score with the revenue term swapped:

`score_adjusted` = `score` − 0.70 × revenue term + 0.70 × fraud-adjusted revenue term

Both terms are computed the same way. Every other term is left exactly as v2 computed it.

In [ ]:
ranking = pd.read_parquet(RANKING_IN)
risk = spark.read.parquet(str(RISK_IN)).select(
    "merchant_abn", "risk_tier", "merchant_score_max", "p_flagged").toPandas()
base = (ranking.merge(merchant_loss, on="merchant_abn", how="left")
               .merge(risk, on="merchant_abn", how="left"))
base["risk_tier"] = base["risk_tier"].fillna("none")

K, REVENUE_WEIGHT = 300, 0.70   # from merchant_ranking_v2.ipynb

def revenue_term(values, n):
    shrinkage = n / (n + K)
    return shrinkage * values.rank(pct=True) * 100 + (1 - shrinkage) * 50

def apply_penalty(min_score, severity):
    r = base.copy()
    r["loss_rate"] = (severity * r[f"loss_min{min_score}"] / r["window_value"]).fillna(0.0)
    r["expected_fraud_loss"] = r["loss_rate"] * r["total_value"]
    r["fraud_adjusted_revenue"] = r["estimated_bnpl_revenue"] - r["expected_fraud_loss"]
    r["loss_to_revenue"] = r["expected_fraud_loss"] / r["estimated_bnpl_revenue"]
    old_term = revenue_term(r["estimated_bnpl_revenue"], r["transaction_count"])
    new_term = revenue_term(r["fraud_adjusted_revenue"], r["transaction_count"])
    r["score_adjusted"] = r["score"] - REVENUE_WEIGHT * old_term + REVENUE_WEIGHT * new_term
    r["rank_adjusted"] = r["score_adjusted"].rank(ascending=False, method="first").astype(int)
    r["rank_change"] = r["rank_overall"] - r["rank_adjusted"]
    return r

# With no loss, the swap must reproduce v2 exactly.
check = apply_penalty(MIN_SCORE_OPTIONS[0], severity=0.0)
assert np.allclose(check["score_adjusted"], check["score"])

### Sensitivity

The scores' calibration is unknown, so the table sweeps two assumptions:

- `MIN_SCORE`: the lowest score treated as a fraud signal. 30 and 50 match the flag tiers.
- `LOSS_SEVERITY`: the share of principal actually lost after recoveries and chargebacks.

For each setting, the table shows how far the ranking moves.

In [ ]:
top_v2 = set(base.nsmallest(100, "rank_overall")["merchant_abn"])
sweep = []
for m in MIN_SCORE_OPTIONS:
    for sev in (1.0, 0.5, 0.25):
        r = apply_penalty(m, sev)
        top = r.nsmallest(100, "rank_adjusted")
        sweep.append({"MIN_SCORE": m, "LOSS_SEVERITY": sev,
                      "Spearman vs v2": spearmanr(r["rank_overall"], r["rank_adjusted"]).statistic,
                      "top-100 overlap": len(top_v2 & set(top["merchant_abn"])),
                      "loss > revenue": int((r["fraud_adjusted_revenue"] < 0).sum()),
                      "median loss/revenue (affected)": r.loc[r["expected_fraud_loss"] > 0, "loss_to_revenue"].median(),
                      "top-100 revenue kept": top["fraud_adjusted_revenue"].sum() / top["estimated_bnpl_revenue"].sum()})
pd.DataFrame(sweep).round(3)

In [ ]:
# Chosen setting - see the sweep above and the notes at the end.
MIN_SCORE, LOSS_SEVERITY = 30, 1.0
r = apply_penalty(MIN_SCORE, LOSS_SEVERITY)
top_new = set(r.nsmallest(100, "rank_adjusted")["merchant_abn"])
print(f"MIN_SCORE={MIN_SCORE}, LOSS_SEVERITY={LOSS_SEVERITY}: top-100 overlap {len(top_v2 & top_new)}, "
      f"Spearman {spearmanr(r['rank_overall'], r['rank_adjusted']).statistic:.3f}")

COLS = ["name", "rank_overall", "rank_adjusted", "estimated_bnpl_revenue", "expected_fraud_loss",
        "loss_to_revenue", "risk_tier", "merchant_score_max"]
print("\nLeft the top 100:")
display(r[r["merchant_abn"].isin(top_v2 - top_new)].sort_values("rank_overall")[COLS].round(3))
print("Entered the top 100:")
display(r[r["merchant_abn"].isin(top_new - top_v2)].sort_values("rank_adjusted")[COLS].round(3))

In [ ]:
print("Biggest penalties among the v2 top 500 (rank places lost):")
r[r["rank_overall"] <= 500].nsmallest(15, "rank_change")[COLS + ["rank_change"]].round(3)

## 6. Review flags on the shortlist

The dollar penalty only covers observed scores. The shortlist also carries the `risk_tier` from `merchant_risk.ipynb`. That includes `suspect`: merchants that were never scored but look like scored ones.

These are **flags for manual review, not exclusions**, in line with treating scores as flags rather than proof. The size confound noted in `merchant_risk.ipynb` applies. `suspect` leans on merchant size, so large merchants are over-represented among suspects.

In [ ]:
top100 = r.nsmallest(100, "rank_adjusted").copy()
top100["fraud_review"] = top100["risk_tier"].isin(["high", "flagged", "suspect"])
print(top100["risk_tier"].value_counts().to_string())
print(f"\n{top100['fraud_review'].sum()} of the fraud-adjusted top 100 are flagged for review")
top100.loc[top100["fraud_review"], ["rank_adjusted", "name", "risk_tier", "merchant_score_max",
                                    "p_flagged", "expected_fraud_loss", "loss_to_revenue"]].round(3)

In [ ]:
OUT_RANK = PROCESSED_PATH / "merchant_ranking_v2_fraud_adjusted.parquet"
OUT_TOP = PROCESSED_PATH / "top100_v2_fraud_adjusted.csv"
r.sort_values("rank_adjusted").to_parquet(OUT_RANK, index=False)
top100.to_csv(OUT_TOP, index=False)
print("written:", OUT_RANK.name, OUT_TOP.name)

## Interpretation and limitations

**Tier classifier**
- Leave-one-merchant-out PR-AUC:
  - ≥ 30: about 0.96, against a random baseline of 0.64.
  - ≥ 50: about 0.91–0.93, against 0.24.
  - ≥ 70: not usable. There are only 7 days, each from a different merchant.
- The score follows how unusual the day is for that merchant, especially a single transaction far above the merchant's usual. This mirrors the consumer score.

**Penalty**
- **Default setting:** `MIN_SCORE = 30` (only flagged scores count) and `LOSS_SEVERITY = 1.0` (full principal lost). That gives Spearman 0.975 against v2, a top-100 overlap of 99, and 99% of the top 100's revenue kept.
- **Mid-table merchants move most.** Merchants with scored days and thin revenue fall hardest. The penalty mostly reorders the middle and tail, not the shortlist.
- **Counting every score is much harsher.** At `MIN_SCORE = 0`, the 8–15% "scored low" days also count. Spearman falls to 0.85, and 553 merchants have expected loss above their revenue. Most of that is big-ticket merchants, because consumer scores follow transaction size.
- **Consumer scores make up more than 95% of expected loss** at every setting, so the penalty is largely a consumer-fraud penalty.
- **The expected loss is a lower bound.** Unscored transactions count as 0, and scores exist only for Feb 2021 – Feb 2022.
- **Calibration of the scores is unknown.** Quote the sweep in the report rather than a single number.
- **Review flags are not in the score.** `risk_tier` from `merchant_risk.ipynb` marks shortlist merchants for review. It is not a penalty, because its `suspect` component leans on merchant size.